# Part C — Prove the Palette, Do Not Claim It (9 marks)

**Objective:** Test two four-colour palettes with the course's `vizlib.simulate_cvd()` and `vizlib.worst_pair()` functions. This notebook calculates the actual worst-pair ΔE values from the course toolkit rather than guessing them.

## 0. Setup
Run in Google Colab. The cell clones the official course repository and imports the provided `vizlib.py`.

In [ ]:
from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex, to_rgb
from IPython.display import display, Markdown

REPO_URL = "https://github.com/UsmarHaider/data-visualization.git"
REPO_DIR = Path("/content/data-visualization")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

candidates = [
    REPO_DIR / "week3" / "assignments",
    REPO_DIR / "week2" / "assignments",
    REPO_DIR,
]
ASSIGN_DIR = next((p for p in candidates if (p / "vizlib.py").exists()), None)
if ASSIGN_DIR is None:
    matches = list(REPO_DIR.rglob("vizlib.py"))
    if not matches:
        raise FileNotFoundError("Could not find course vizlib.py in cloned repository.")
    ASSIGN_DIR = matches[0].parent
sys.path.insert(0, str(ASSIGN_DIR))
import vizlib

print("Repository:", REPO_DIR)
print("Toolkit:", ASSIGN_DIR / "vizlib.py")
print("Course palette keys:", list(vizlib.PALETTE.keys()))
print("Course series palette:", vizlib.PALETTE["series"])

## 1. Choose a palette without assuming it is safe (3 marks)

For the comparison palette, I chose the first four colours from Matplotlib's `tab10`. It is widely used and visually varied, but popularity is not proof of accessibility.

In [ ]:
palette_a = [to_hex(c) for c in plt.get_cmap("tab10").colors[:4]
]
palette_b = list(vizlib.PALETTE["series"]) + [vizlib.PALETTE["series_4th"]]

print("Palette A — Matplotlib tab10, first four colours:")
display(pd.DataFrame({"category": ["Category 1","Category 2","Category 3","Category 4"],
                      "hex": palette_a}))
print("Palette B — course series plus fourth slot:")
display(pd.DataFrame({"category": ["Category 1","Category 2","Category 3","Category 4"],
                      "hex": palette_b}))

### Before/after swatches

Each row shows the same four category colours before simulation and after a simulated type of colour-vision deficiency. The simulation uses the course implementation, not a hand-picked approximation.

In [ ]:
def draw_swatches(ax, colours, title):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 4)
    ax.set_title(title, fontsize=11, pad=8)
    ax.axis("off")
    for i, colour in enumerate(colours):
        y = 3-i
        ax.add_patch(plt.Rectangle((0.04, y+0.12), 0.25, 0.72,
                                   color=colour, transform=ax.transData))
        ax.text(0.34, y+0.48, f"Category {i+1}  {to_hex(colour)}",
                va="center", fontsize=9)

def show_palette_simulations(palette, palette_name, filename):
    kinds = ["normal", "protan", "deutan", "tritan"]
    fig, axes = plt.subplots(4, 2, figsize=(10, 9))
    for row, kind in enumerate(kinds):
        if kind == "normal":
            simulated = palette
            label = "Original (no simulation)"
        else:
            simulated = vizlib.simulate_cvd(palette, kind)
            label = f"Simulated {kind}"
        draw_swatches(axes[row, 0], palette, "Original palette" if row == 0 else "")
        draw_swatches(axes[row, 1], simulated, label)
    fig.suptitle(f"{palette_name}: original and CVD simulations", fontsize=14, y=1.01)
    fig.tight_layout()
    output = Path("/content") / filename
    fig.savefig(output, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", output)

show_palette_simulations(palette_a, "Palette A — Matplotlib tab10", "partC_tab10_cvd.png")

**Part 1 interpretation:** Compare each simulated row with the original. Look specifically for pairs that become similar in hue or lightness. A four-colour palette can look clear to a viewer with typical colour vision and still lose category distinctions under simulation.

## 2. Test the course palette plus its fourth slot (2 marks)

In [ ]:
show_palette_simulations(palette_b, "Palette B — course palette plus fourth slot", "partC_course_palette_cvd.png")

## 3. Find the worst pair and report ΔE (2 marks)

`vizlib.worst_pair()` reports the pair with the smallest perceptual separation for each simulated vision type. The notebook reports the returned colour indices, hex colours, and ΔE. Use the course's `vizlib.verdict()` for the interpretation threshold.

In [ ]:
def audit_palette(palette, label):
    records = []
    for vision in ["normal", "protan", "deutan", "tritan"]:
        i, j, c1, c2, delta_e = vizlib.worst_pair(palette, vision)
        records.append({
            "palette": label,
            "vision simulation": vision,
            "worst pair (indices)": f"{i} and {j}",
            "colour 1": to_hex(c1),
            "colour 2": to_hex(c2),
            "ΔE": float(delta_e),
            "course verdict": vizlib.verdict(delta_e),
        })
    return records

results = pd.DataFrame(
    audit_palette(palette_a, "Matplotlib tab10 (first four)") +
    audit_palette(palette_b, "Course palette + fourth slot")
)
display(results.round({"ΔE": 3}))
results.to_csv("/content/partC_palette_audit.csv", index=False)
print("Saved /content/partC_palette_audit.csv")

### Interpretation to include with the table

- The **smallest ΔE** is the pair most likely to be confused under that simulation. Use the actual returned values and the course verdict in your report.
- If the course palette's fourth slot collapses with another colour, it is **not permitted to function as the only signal** distinguishing a fourth category. Keep the fourth category identifiable through direct text labels and a redundant visual channel such as marker shape, line style, hatching, or a small labelled table.
- Simulation is a useful diagnostic, not a guarantee that every viewer, display, or viewing condition will behave identically.

## 4. Rule for the rest of the course (2 marks)

**My rule:** I will use no more than three category colours when colour is the only category cue. If a chart needs more than three categories, I will not solve the problem by adding more colours. I will use direct labels and add a redundant channel—such as marker shape, line style, hatching, or small multiples—so categories remain identifiable even when colours collapse under CVD simulation. I will check the legend and the chart in greyscale as well as under the course's protan, deutan, and tritan simulations.

The course palette's fourth slot will only be used with a redundant cue and a clear label, never as a colour-only distinction.

## Files produced

- `partC_tab10_cvd.png` — before/after swatches for Matplotlib tab10.
- `partC_course_palette_cvd.png` — before/after swatches for the course palette plus fourth slot.
- `partC_palette_audit.csv` — worst-pair ΔE and verdict for normal, protan, deutan, and tritan vision.

**Submission reminder:** Run all cells and paste the actual table values and swatch figures into your assignment. Do not submit unexecuted placeholder results.